In [3]:
import torch
import torch.nn as nn

In [4]:
def intersection_over_union(boxes_preds, boxes_labels, is_yolo:bool=True):
    """
    Calculates Intersection over Union (IoU) with support for broadcasting.
    
    Args:
        boxes_preds (Tensor): Predicted boxes, shape [..., num_boxes, 4]
        boxes_labels (Tensor): Target boxes, shape [..., 1, 4]
        box_format (str): 'midpoint' (x,y,w,h) or 'corners' (x1,y1,x2,y2)
        
    Returns:
        Tensor: IoU for all pairs, shape [..., num_boxes]
    """
    if is_yolo:
        # Convert [x, y, w, h] to [x1, y1, x2, y2]
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2
        
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    else:
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]
        
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    # 1. Calculate intersection coordinates using element-wise max/min
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # 2. Compute intersection area (.clamp(0) handles cases with zero overlap)
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    # 3. Compute individual box areas
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    # 4. Calculate union area (Area A + Area B - Intersection)
    union = box1_area + box2_area - intersection

    # 5. Return IoU score (1e-6 prevents dividing by absolute zero)
    return (intersection / (union + 1e-6)).squeeze(-1)


For every grid cell in an $S \times S$ grid ($S=7$), YOLOv1 outputs a tensor of size **30** (where $B=2$ bounding boxes and $C=20$ PASCAL VOC classes):

$$\text{Vector Size per Cell} = C + B \times 5 = 20 + (2 \times 5) = 30$$

### Vector Layout Breakdown:
* **Indices `0 to 19` (20 values):** Class probabilities ($c_1, c_2, \dots, c_{20}$).
* **Index `20` (1 value):** Confidence score ($p_{c1}$) for Bounding Box 1.
* **Index `21` (1 value):** Confidence score ($p_{c2}$) for Bounding Box 2.
* **Indices `22 to 25` (`21:25`) (4 values):** Coordinates $(x, y, w, h)$ for Bounding Box 1.
* **Indices `26 to 29` (`26:30`) (4 values):** Coordinates $(x, y, w, h)$ for Bounding Box 2.

### Code Operations:
* `predictions[..., 22:26]`: Slices out the $(x, y, w, h)$ coordinates for **Box 1**.
* `predictions[..., 27:30]`: Slices out the $(x, y, w, h)$ coordinates for **Box 2**.
* `target[..., 21:25]`: Slices out the ground-truth target coordinates.

---


In [ ]:
class YoloV1Loss(nn.Module):
    def __init__(self, S:int=7, B:int=2, C:int=1):
        super(YoloV1Loss, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        self.mse = nn.MSELoss(reduction="sum")
        self.lam_no_obj = 0.5
        self.lam_obj = 5
    def forward(self, predictions:torch.Tensor, target:torch.Tensor):
        """
        predictions will have (num_batches, S*S*(B*5 + C))
        target will have (num_batches, S, S, C + 1*5)
        """
        predictions = predictions.reshape(-1, self.S, self.S, self.B * 5 + self.C)
        start_index = self.C + self.B
        all_pred_boxes = predictions[..., start_index:(start_index + self.B*4)].reshape(*predictions.shape[:-1], self.B, 4)
        target_box = target[..., (self.C+1):(self.C+5)].unsqueeze(-2)
        ious = intersection_over_union(all_pred_boxes, target_box)
        ious_max = torch.argmax(ious, dim=-1)
        best_box_expanded = ious_max.unsqueeze(-1).unsqueeze(-1)

        # For torch.gather to work the index must have the same shape as input
        exists_box = target[..., self.C, None, None]
        best_pred_box = torch.gather(all_pred_boxes, dim=-2, index=best_box_expanded)

        # To calculate box loss
        # 2:4 height and width
        # Use box_loss = (sqrt(sqrt(w1) - sqrt(w2))^2 + (sqrt(h1) - sqrt(h2))^2)
        pred_xy = best_pred_box[..., 0:2]
        pred_hw = torch.sign(best_pred_box[..., 2:4])*torch.sqrt(torch.abs(best_pred_box[..., 2:4]))
        transformed_pred = torch.cat([pred_xy, pred_hw], dim=-1)
        
        target_xy = target_box[..., 0:2]
        target_wh = torch.sqrt(target_box[..., 2:4])
        transformed_targets = torch.cat([target_xy, target_wh], dim=-1)

        box_predictions = exists_box*transformed_pred
        box_targets = exists_box*transformed_targets

        box_loss = self.mse(
            torch.flatten(box_predictions, end_dim=-2),
            torch.flatten(box_targets, end_dim=-2),
        )

        # To calculate probability of box

        all_pred_prob = predictions[..., (self.C):(self.C + self.B)].reshape(*predictions.shape[:-1], self.B) # shape is [batch, S, S, 1]
        best_pred_prob = torch.gather(all_pred_prob, dim=-1, index=ious_max.unsqueeze(-1))
        exists_prob = exists_box.squeeze(-1) # exists_prob is [batches, S, S, 1]
        target_prob = target[..., self.C:(self.C+1)]
        object_loss = self.mse(
            torch.flatten(exists_prob*best_pred_prob),
            torch.flatten(exists_prob*target_prob)
        )

        # To calculate no object loss

        no_object_target = torch.zeros_like(all_pred_prob)

        responsible_box_mask = nn.functional.one_hot(ious_max, num_classes=self.B)
        no_object_mask = 1- exists_prob*responsible_box_mask

        masked_object_pred = no_object_mask*all_pred_prob
        masked_object_target = no_object_mask*no_object_target

        no_object_loss = self.mse(
            torch.flatten(masked_object_pred),
            torch.flatten(masked_object_target)
        )

        # To calculate Class Loss

        class_loss = self.mse(
            torch.flatten(exists_prob * predictions[..., :self.C], end_dim=-2),
            torch.flatten(exists_prob * target[..., :self.C], end_dim=-2),
        )
        
        # TOTAL LOSS
        loss = (
            self.lam_obj * box_loss
            + object_loss
            + self.lam_no_obj * no_object_loss
            + class_loss
        )
        return loss

In [45]:
num_batches = 5
S = 7
B = 2
C = 1
mse = nn.MSELoss(reduction="sum")

predictions = torch.rand(num_batches, S*S*(5*B+C))
target = torch.rand(num_batches, S, S, 5*1+C)

predictions = predictions.reshape(-1, S, S, B * 5 + C)
start_index = C + B
all_pred_boxes = predictions[..., start_index:(start_index + B*4)].reshape(*predictions.shape[:-1], B, 4)
target_box = target[..., (C+1):(C+5)].unsqueeze(-2)
ious = intersection_over_union(all_pred_boxes, target_box)
ious_max = torch.argmax(ious, dim=-1)
best_box_expanded = ious_max.unsqueeze(-1).unsqueeze(-1)

# For torch.gather to work the index must have the same shape as input
exists_box = target[..., C, None, None]
best_pred_box = torch.gather(all_pred_boxes, dim=-2, index=best_box_expanded)

# To calculate box loss
# 2:4 height and width
# Use box_loss = (sqrt(sqrt(w1) - sqrt(w2))^2 + (sqrt(h1) - sqrt(h2))^2)
pred_xy = best_pred_box[..., 0:2]
pred_hw = torch.sign(best_pred_box[..., 2:4])*torch.sqrt(torch.abs(best_pred_box[..., 2:4]))
transformed_pred = torch.cat([pred_xy, pred_hw], dim=-1)

target_xy = target_box[..., 0:2]
target_wh = torch.sqrt(target_box[..., 2:4])
transformed_targets = torch.cat([target_xy, target_wh], dim=-1)

box_predictions = exists_box*transformed_pred
box_targets = exists_box*transformed_targets

box_loss = mse(
    torch.flatten(transformed_pred, end_dim=-2),
    torch.flatten(transformed_targets, end_dim=-2),
)

# To calculate probability of box

all_pred_prob = predictions[..., (C):(C + B)].reshape(*predictions.shape[:-1], B)
best_pred_prob = torch.gather(all_pred_prob, dim=-1, index=ious_max.unsqueeze(-1))

exists_prob = exists_box.squeeze(-1) # exists_box is now [batches, S, S, 1]
target_prob = target[..., C:(C+1)]
object_loss = mse(
    torch.flatten(exists_prob*best_pred_prob),
    torch.flatten(exists_prob*target_prob)
)


no_object_target = torch.zeros_like(all_pred_prob)

responsible_box_mask = nn.functional.one_hot(ious_max, num_classes=B)
no_object_mask = 1- exists_prob*responsible_box_mask

masked_object_pred = no_object_mask*all_pred_prob
masked_object_target = no_object_mask*no_object_target

no_object_loss = mse(
    torch.flatten(masked_object_pred),
    torch.flatten(masked_object_target)
)

class_loss = mse(
    torch.flatten(exists_prob * predictions[..., :C], end_dim=-2),
    torch.flatten(exists_prob * target[..., :C], end_dim=-2),
)

loss = (
    5 * box_loss
    + object_loss
    + 0.5 * no_object_loss
    + class_loss
)
loss
        

/home/silverhorn/Desktop/Learning Codes/Face Recognition/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([245, 4])) that is different to the input size (torch.Size([245, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


tensor(726.8064)